# Libraries

Just like before, lets import the libraries we'll be using. In addition to our usual libraries, we'll also import the python file we used to save our data processing functions from our source code directory (src). We can import it like a module since we've added an __init__.py file within that folder alongside adding it to our system's path list.

In [5]:
import pandas as pd
import sys
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import mlflow
import json

sys.path.append(os.path.abspath(os.pardir))

from src.data_processing import *

# Dataset Processing

Here, we'll take a look at the processed dataset. Fortunately, we've already done the data processing in our data_processing python notebook, all we need to do now is to call the process_data() function we've prepared inside our data_processing python file.

In [ ]:
dataset_path = os.path.join('..','datasets','Philippine Fake News Corpus.csv')
df = pd.read_csv(dataset_path)
train, test = process_data(
    df = df, 
    feature_col = 'Content',
    label_col = 'Label',
    random_state = 42,
    df_name = 'Philippine Fake News Corpus.csv'
)

display(train)
display(test)

,Content,Label
0,"[397, 398, 7972, 5154, 8, 2589, 3706, 5252, 10...",1
1,"[542, 7972, 1242, 2722, 4, 1841, 2654, 6506, 7...",0
2,"[542, 7972, 196, 7075, 29, 1582, 7953, 7926, 5...",0
3,"[397, 398, 7972, 220, 4354, 137, 7964, 7934, 1...",1
4,"[397, 398, 7972, 220, 4126, 73, 6316, 790, 795...",1
...,...,...
23678,"[311, 967, 7972, 220, 2905, 2615, 27, 8, 295, ...",0
23679,"[311, 846, 7972, 220, 3219, 2805, 1639, 7926, ...",0
23680,"[542, 7972, 220, 1133, 65, 3940, 137, 5609, 79...",0
23681,"[1716, 1715, 7972, 1055, 1063, 5485, 209, 65, ...",1


,Content,Label
0,"[4020, 4110, 295, 7972, 2129, 1433, 1346, 7941...",1
1,"[1716, 1715, 7972, 716, 150, 1523, 73, 64, 566...",1
2,"[397, 398, 7972, 7419, 358, 76, 426, 6153, 65,...",1
3,"[542, 7972, 220, 1475, 133, 8, 774, 27, 1148, ...",0
4,"[397, 398, 7972, 7233, 926, 7941, 220, 397, 39...",1
...,...,...
5916,"[4020, 4110, 295, 7972, 220, 2800, 201, 853, 3...",1
5917,"[542, 7972, 57, 2330, 205, 2467, 63, 1366, 450...",0
5918,"[397, 398, 7972, 7419, 347, 1290, 466, 5836, 7...",1
5919,"[1716, 1715, 7972, 57, 420, 190, 4242, 5202, 7...",1


Our function looks functional, separating our dataset into train and test in addition to encoding them into numeric representations that we can work with. However, before we can pass this to the model, it needs to be arranged into tensors.

In addition to this, we want to pass it by batches, as such, we'll have to configure a dataloader for our model.

# Dataloader

Our dataloader will convert our dataset into batches, a sample of the actual dataset that our model can learn with in increments. Before this, we'll have to convert it to tensors, the shapes of the tensors must be consistent, which means that we'll have to pad all the sequences to have the same length.

## Converting to Tensor and Padding

Before creating a custom dataset, we need to make our values into tensors with consistent shape. Making a consistent shape can be done through padding, luckily for us, pytorch already has a dedicated function for this. All we have to do now is to convert the values into tensors.

In [ ]:
train.Content = train.Content.apply(lambda x: torch.tensor(x))
train.Label = train.Label.apply(lambda x: torch.tensor(x))

test.Content = test.Content.apply(lambda x: torch.tensor(x))
test.Label = test.Label.apply(lambda x: torch.tensor(x))

display(train.head())
display(test.head())

,Content,Label
0,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)
1,"[tensor(542), tensor(7972), tensor(1242), tens...",tensor(0)
2,"[tensor(542), tensor(7972), tensor(196), tenso...",tensor(0)
3,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)
4,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)


,Content,Label
0,"[tensor(4020), tensor(4110), tensor(295), tens...",tensor(1)
1,"[tensor(1716), tensor(1715), tensor(7972), ten...",tensor(1)
2,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)
3,"[tensor(542), tensor(7972), tensor(220), tenso...",tensor(0)
4,"[tensor(397), tensor(398), tensor(7972), tenso...",tensor(1)


After converting to tensors, we want to pad them. We can make use of the pad_sequence() function from pytorch to pad them to the max length in the dataset; however, if we do this separately for each dataset, we'll end up with two different lengths: one max for train and another for test. To resolve this, we'll have to briefly combine train and test, then pad them, afterwards, separate them when calling the custom Dataset class.

In [ ]:
merged = pd.concat([
    train.Content,
    test.Content
])

padded = pad_sequence(merged, batch_first = True, padding_value = 3)
padded.shape

torch.Size([29604, 24186])

## Creating a Custom Dataset

Before we can make use of a dataloader, we need to create a custom dataset class that the dataloader can work with. Creating it is quite simple, as we simply need to create a pseudo-custom dataset class that has the basic dunder methods like indexing and len.

In [ ]:
class FakeNewsDataset(Dataset):
    def __init__(self, feature, label):
        super().__init__()
        self.feature = feature
        self.label = label
        
    def __len__(self):
        return len(self.feature)
    
    def __getitem__(self, idx):
        feature = self.feature[idx]
        label = self.label[idx]
        
        return feature, label

In [ ]:
fake_news_train = FakeNewsDataset(
    padded[:len(train),:],
    train.Label
)

fake_news_test = FakeNewsDataset(
    padded[len(train):, :],
    test.Label
)

In [ ]:
fake_news_train[0], fake_news_train[0][0].shape

((tensor([ 397,  398, 7972,  ...,    3,    3,    3]), tensor(1)),
 torch.Size([24186]))

In [ ]:
fake_news_test[0], fake_news_test[0][0].shape

((tensor([4020, 4110,  295,  ...,    3,    3,    3]), tensor(1)),
 torch.Size([24186]))

## Creating the Dataloader

In [9]:
train_loader = DataLoader(
    dataset = fake_news_train,
    batch_size = 32,
    shuffle = True
)

test_loader = DataLoader(
    dataset = fake_news_test,
    batch_size = 32,
    shuffle = True
)

# Model Architecture

Now that we've dealt with our dataset, all that's left is to design our model architecture. Before anything, we'll start with an embedding layer so that we can convert the indices into context vectors that the model can use to learn meaning. We'll use a similar architecture to a CNN, using 1 dimensional convolutional layers paired with max pooling layers. Afterwards, it will be passed to a flatten layer and finally to a linear layer for the final prediction.

Defining the parameters for the layers are quite simple, with the sole exception of the linear layer. The linear layer's input shape is wholly dependent on the output of the flattened output from the convolutional and pooling layers, but the way the output shape is defined in these layers are through a complex formula. One thing we can do to resolve this is to simulate a single forward pass through the layers until the flatten layer, then extract the size of that layer. We then plug that into the linear definition as the input shape. As this is only a simulation, we need to define this under a no gradient context so that the code recognizes to not build a gradient or treat it as a training input.

In [10]:
class FakeNewsDetector(nn.Module):
    def __init__(self, vocab_size, embed_dim, pad_id, conv_dim, kernel_size, max_seq):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, pad_id)
        self.conv1d = nn.Conv1d(embed_dim, conv_dim, kernel_size)
        self.pool1d = nn.MaxPool1d(kernel_size)
        self.flat = nn.Flatten()
        
        # Calculate Flattened shape
        with torch.no_grad():
            dummy = torch.zeros(1, max_seq, dtype = torch.int64)
            embed = self.embed(dummy)
            conv1d = self.conv1d(
                embed.transpose(1,2)
            )
            pool1d = self.pool1d(conv1d)
            flat = self.flat(pool1d)
            flattened_size = flat.size(1)
        
        self.fc = nn.Linear(flattened_size, 1)
        
    def forward(self, x):
        x = self.embed(x)
        x = x.transpose(1,2)
        x = self.conv1d(x)
        x = self.pool1d(x)
        x = self.flat(x)
        x = self.fc(x)
        return x

# Model Training

After defining the model, we'll train the model by defining the optimizer and the loss function. We will also log the model's parameters, dataset, and other metrics so that we can reproduce this same exact model in the future if ever we need to revisit this exact version of the model.

In [11]:
max_seq = fake_news_train[0][0].shape[0]
max_seq

24186

In [ ]:
config = {
    'vocab_size': 8000,
    'embed_dim': 5,
    'pad_id': 3,
    'conv_dim': 4,
    'kernel_size': 5,
    'max_seq': max_seq,
}

model = FakeNewsDetector(**config)

In [27]:
optim = torch.optim.Adam(model.parameters(), lr = 0.001)
loss_func = nn.BCEWithLogitsLoss()

## Setting Up MLFlow Tracking

In [ ]:
model_path = os.path.join('..','models','FakeNewsDetector')

mlflow.set_tracking_uri('sqlite:///' + os.path.join(os.path.abspath(model_path),'mlflow.db'))

experiment = mlflow.get_experiment_by_name('FakeNewsDetector')

if experiment is None:
    experiment_id = mlflow.create_experiment(
        name = 'FakeNewsDetector',
        artifact_location = model_path
    )
else:
    experiment_id = experiment.experiment_id

curr_experiment = mlflow.set_experiment(experiment_id = experiment_id)
print(f'The current active experiment has been set to {curr_experiment.name} with id of {curr_experiment.experiment_id}.')

The current active experiment has been set to FakeNewsDetector with id of 1


## Training Loop

In [28]:
epochs = 3

with mlflow.start_run():
    
    # Log Parameters
    mlflow.log_params(config)
    
    for epoch in range(epochs):
        # Accumulate Loss
        epoch_loss = 0
        
        for batch in train_loader:
            X = batch[0]
            y = batch[1].to(torch.float32)
            
            logits = model(X)
            loss = loss_func(
                logits.reshape(-1),
                y.to(torch.float32)
            )
            
            epoch_loss += loss.item()
            
            optim.zero_grad()
            loss.backward()
            optim.step()
        
        # Track Loss
        mlflow.log_metric('Loss', epoch_loss, step = epoch)
        
        print(f'Epoch: {epoch} | Loss: {round(epoch_loss/len(train_loader), 4)}')
    
    # Temporary Weights File
    torch.save(
        model.state_dict(),
        'weights.pt'
    )
    
    # Temporary Config File
    with open('config.json', 'w') as f:
        json.dump(config, f, indent = 2)
    
    # Log temp files
    mlflow.log_artifact('weights.pt')
    mlflow.log_artifact('config.json')
    
    # Remove temporary files
    os.remove('weights.pt')
    os.remove('config.json')

Epoch: 0 | Loss: 0.4325
Epoch: 1 | Loss: 0.021
Epoch: 2 | Loss: 0.0034


# Tests

In [ ]:
all_runs = mlflow.search_runs()
all_runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.Loss,params.embed_dim,params.pad_id,params.vocab_size,params.kernel_size,params.max_seq,params.conv_dim,tags.mlflow.user,tags.mlflow.source.name,tags.mlflow.runName,tags.mlflow.source.type
0,225bddb08550475ba1bd78085bf05958,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-28 10:55:44.448000+00:00,2026-05-28 11:01:53.473000+00:00,2.548278,5,3,8000,5,24186,4,kayle,model_building.ipynb,caring-bee-173,NOTEBOOK
1,4f52c08b60cd4140bc8935a793c777e1,1,FAILED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-28 10:52:06.575000+00:00,2026-05-28 10:55:34.380000+00:00,0.047758,5,3,8000,5,24186,4,kayle,model_building.ipynb,suave-bug-402,NOTEBOOK
2,6af49473c2144876856f7187fb6f0279,1,FAILED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-28 10:43:57.105000+00:00,2026-05-28 10:49:56.884000+00:00,0.077309,5,3,8000,5,24186,4,kayle,model_building.ipynb,dazzling-shark-675,NOTEBOOK
3,956bb4df39604c7392e56c2b9d4d0af5,1,FAILED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-28 10:27:35.336000+00:00,2026-05-28 10:34:32.406000+00:00,0.667525,5,3,8000,5,24186,4,kayle,model_building.ipynb,monumental-jay-746,NOTEBOOK
4,d6e144ce946741a7a71f281492be4e6b,1,FAILED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-28 10:27:12.317000+00:00,2026-05-28 10:27:21.303000+00:00,NaN,5,3,8000,5,24186,4,kayle,model_building.ipynb,thoughtful-shoat-832,NOTEBOOK
5,6610f5fc36e2420f9bd8569144e9c608,1,FAILED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-28 10:21:38.515000+00:00,2026-05-28 10:25:00.604000+00:00,318.631692,5,3,8000,5,24186,4,kayle,model_building.ipynb,luxuriant-midge-208,NOTEBOOK


In [39]:
finished_runs = mlflow.search_runs(filter_string = "status = 'FINISHED'")
finished_runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.Loss,params.embed_dim,params.pad_id,params.vocab_size,params.kernel_size,params.max_seq,params.conv_dim,tags.mlflow.user,tags.mlflow.source.name,tags.mlflow.runName,tags.mlflow.source.type
0,225bddb08550475ba1bd78085bf05958,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-28 10:55:44.448000+00:00,2026-05-28 11:01:53.473000+00:00,2.548278,5,3,8000,5,24186,4,kayle,model_building.ipynb,caring-bee-173,NOTEBOOK


In [43]:
os.listdir(os.path.join(model_path, finished_runs.run_id[0], 'artifacts'))

['config.json', 'weights.pt']

In [ ]:
torch.load(os.path.join(model_path, finished_runs.run_id[0], 'artifacts', 'weights.pt'))